<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/04_sentinel_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sentinel and vegetation features

**Purpose:** Add satellite-derived vegetation, moisture and soil-surface features to every grid cell from notebook 01, for use as model inputs.

**Inputs:**
- `grid.parquet` from notebook 01 (~2 km cells, nine Sahel countries)
- Google Earth Engine: Sentinel-2 SR Harmonized (`COPERNICUS/S2_SR_HARMONIZED`) with Cloud Score+ masking (`GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED`), Sentinel-1 GRD (`COPERNICUS/S1_GRD`), MODIS NDVI (`MODIS/061/MOD13Q1`)
- Requires an Earth Engine account and a Cloud project (set `PROJECT_ID` to your own)

**Method:** Features are computed on Earth Engine as typical-year composites over 2019-2023 (one median per calendar month, pooled across years), built at 250 m and averaged to the 2 km grid (validated against a 100 m version; see the resolution test result), sampled at each grid point and exported as small CSV tiles, then merged here.
- **Sentinel-2:** monthly NDVI (12 features), wet-season NDWI, dry-season salinity index (bare-soil proxy)
- **Sentinel-1:** wet- and dry-season VV and VH backscatter (dB), wet-season VH-VV difference, wet-season VV variability
- **Derived in pandas after the merge:** peak NDVI, peak month, amplitude, wet- and dry-season means
- **MODIS:** long-term NDVI trend (2001-2024) as a land-degradation indicator

**Outputs** (stored in Google Drive, not in this repo): `sentinel.parquet`, one row per `grid_id` with 30 float32 features:
- 20 exported from Earth Engine (12 monthly NDVI, wet-season NDWI, dry-season salinity index, 6 Sentinel-1 radar features)
- 6 derived here (NDVI maximum, minimum, amplitude, peak month, wet-season mean, dry-season mean)
- 4 MODIS growing-season NDVI features (2001-2024 mean, variability, trend and change)

**Notes:**
- Composites describe a typical year and not individual years.
- The Sentinel-2 salinity index is an indirect proxy for surface soil conditions, not measured salinity.
- Sentinel-1 values are averaged in dB for simplicity; edge-noise artefacts may remain.
- Cells with no valid observation in a month are NaN; their share is reported after the merge.
- The 250 m resolution was validated on a test square; a 10 m version exceeded Earth Engine’s memory limits.
- MODIS growing-season NDVI variability (CV) is set to NaN where mean NDVI is below 0.1 or the CV falls outside 0-1 (about 0.4% of cells), because the ratio is unreliable for water, bare sand and very sparse vegetation.

In [ ]:
# @title Cell 1: setup
import ee, os, glob, time
import numpy as np
import pandas as pd
from google.colab import drive

PROJECT_ID = 'your-project-id'
YEARS = (2019, 2023)                  # inclusive; reduce to (2021, 2022) if the test is too heavy
CS_THRESH = 0.6                       # Cloud Score+ threshold (higher = stricter cloud masking)
OUT_FOLDER = 'crop_project_sentinel'
GRID_SCALE = 2000                     # ~2 km grid scale in meters

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive'
grid = pd.read_parquet(f'{BASE}/crop_project/grid.parquet')
print(grid.shape)

proj = ee.Projection('EPSG:4326').atScale(GRID_SCALE)

def to_fc(d):
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {'grid_id': r.grid_id})
        for r in d.itertuples()])

In [ ]:
# @title Cell 2: inspect the datasets
d0, d1 = f'{YEARS[0]}-01-01', f'{YEARS[1] + 1}-01-01'
box = ee.Geometry.Rectangle([8, 12, 9, 13])     # a 1-degree box near Kano

def date_of(col, ascending):
    img = col.limit(1, 'system:time_start', ascending).first()
    return ee.Date(img.get('system:time_start')).format('YYYY-MM-dd').getInfo()

for cid in ['COPERNICUS/S2_SR_HARMONIZED',
            'GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED',
            'COPERNICUS/S1_GRD',
            'MODIS/061/MOD13Q1']:
    col = ee.ImageCollection(cid)
    local = col.filterBounds(box)
    print(cid, '| in test box:', date_of(local, True), '->', date_of(local, False))
    print('   bands:', col.first().bandNames().getInfo()[:14])

print('\nScenes over the test box,', d0, 'to', d1)
print('  S2:', ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(box).filterDate(d0, d1).size().getInfo())
print('  S1:', ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(box).filterDate(d0, d1).size().getInfo())

In [ ]:
# @title Cell 3: the feature builder
NDVI_BANDS = [f'ndvi_m{m:02d}' for m in range(1, 13)]
S2_OTHER   = ['ndwi_wet', 'si_dry']
S1_BANDS   = ['vv_wet', 'vh_wet', 'vv_dry', 'vh_dry', 'vv_wet_std', 'vhvv_wet']

def finish(img, scale_m):
    """Average from scale_m to the 2 km grid; -9999 marks masked pixels."""
    P = ee.Projection('EPSG:4326').atScale(scale_m)
    max_pix = int((GRID_SCALE / scale_m) ** 2 * 1.5)
    return (img.float().setDefaultProjection(P)
            .reduceResolution(ee.Reducer.mean(), maxPixels=max_pix)
            .reproject(proj).unmask(-9999))

def s2_index_collection(region):
    d0, d1 = f'{YEARS[0]}-01-01', f'{YEARS[1] + 1}-01-01'
    s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(region).filterDate(d0, d1))
    cs = (ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')
          .filterBounds(region).filterDate(d0, d1))
    s2 = s2.linkCollection(cs, ['cs_cdf'])

    def prep(img):
        img = img.updateMask(img.select('cs_cdf').gte(CS_THRESH))
        ndvi = img.normalizedDifference(['B8', 'B4']).rename('ndvi')
        ndwi = img.normalizedDifference(['B3', 'B8']).rename('ndwi')
        si = (img.select('B3').multiply(img.select('B4')).divide(1e8)
              .sqrt().rename('si'))
        return (ndvi.addBands([ndwi, si])
                .copyProperties(img, ['system:time_start']))   # <- the fix
    return s2.map(prep)

def s2_ndvi(region, scale_m, months=range(1, 13)):
    idx = s2_index_collection(region)
    bands = [idx.filter(ee.Filter.calendarRange(m, m, 'month')).select('ndvi')
             .median().rename(f'ndvi_m{m:02d}') for m in months]
    return finish(ee.Image.cat(bands), scale_m)

def s2_other(region, scale_m):
    idx = s2_index_collection(region)
    ndwi_wet = (idx.filter(ee.Filter.calendarRange(7, 9, 'month')).select('ndwi')
                .median().rename('ndwi_wet'))
    dry = ee.Filter.Or(ee.Filter.calendarRange(11, 12, 'month'),
                       ee.Filter.calendarRange(1, 3, 'month'))
    si_dry = idx.filter(dry).select('si').median().rename('si_dry')
    return finish(ee.Image.cat([ndwi_wet, si_dry]), scale_m)

def s1_feats(region, scale_m):
    d0, d1 = f'{YEARS[0]}-01-01', f'{YEARS[1] + 1}-01-01'
    s1 = (ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(region).filterDate(d0, d1)
          .filter(ee.Filter.eq('instrumentMode', 'IW'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
          .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
          .select(['VV', 'VH']))
    wet = s1.filter(ee.Filter.calendarRange(7, 9, 'month'))
    dry = s1.filter(ee.Filter.Or(ee.Filter.calendarRange(12, 12, 'month'),
                                 ee.Filter.calendarRange(1, 2, 'month')))
    vv_wet = wet.select('VV').mean().rename('vv_wet')
    vh_wet = wet.select('VH').mean().rename('vh_wet')
    vv_dry = dry.select('VV').mean().rename('vv_dry')
    vh_dry = dry.select('VH').mean().rename('vh_dry')
    vv_std = wet.select('VV').reduce(ee.Reducer.stdDev()).rename('vv_wet_std')
    vhvv = vh_wet.subtract(vv_wet).rename('vhvv_wet')
    return finish(ee.Image.cat([vv_wet, vh_wet, vv_dry, vh_dry, vv_std, vhvv]), scale_m)

print('Builders defined: 12 + 2 + 6 = 20 features')

In [ ]:
# @title Cell 4: resolution test
# compare Sentinel features built at 250 m (B) and 100 m (C)
# on a small test square (0.3 degrees near Kano, northern Nigeria).
# A 10 m version exceeded Earth Engine's memory limit and was dropped.
OUT_FOLDER = 'crop_project_sentinel'
SQ = (8.0, 12.0, 8.3, 12.3)

sq_pts = grid[(grid.lon >= SQ[0]) & (grid.lon < SQ[2]) &
              (grid.lat >= SQ[1]) & (grid.lat < SQ[3])]
region = ee.Geometry.Rectangle(list(SQ))
fc = to_fc(sq_pts)
print(len(sq_pts), 'grid cells in the test square')

for tag, fn in {'s1': s1_feats, 's2oth': s2_other, 'ndvi': s2_ndvi}.items():
    for variant, scale_m in (('B', 250), ('C', 100)):
        ee.batch.Export.table.toDrive(
            collection=fn(region, scale_m).sampleRegions(
                collection=fc, properties=['grid_id'],
                scale=GRID_SCALE, projection=proj, tileScale=8),
            description=f's4t_{tag}_{variant}', folder=OUT_FOLDER, fileFormat='CSV').start()
print('Started 6 test tasks')

In [ ]:
# @title Cell 5: compare the test results
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'

def load(tag, variant):
    f = f'{BASE}/{OUT_FOLDER}/s4t_{tag}_{variant}.csv'
    return (pd.read_csv(f).drop(columns=['system:index', '.geo'], errors='ignore')
            .set_index('grid_id').replace(-9999, np.nan))

tags = ['ndvi', 's2oth', 's1']
B = pd.concat([load(t, 'B') for t in tags], axis=1)     # 250 m
C = pd.concat([load(t, 'C') for t in tags], axis=1)     # 100 m
ix = B.index.intersection(C.index)

comp = pd.DataFrame({
    'mean_250m': B.loc[ix].mean(),
    'mean_100m': C.loc[ix].mean(),
    'correlation': B.loc[ix].corrwith(C.loc[ix]),
    'mean_abs_diff': (B.loc[ix] - C.loc[ix]).abs().mean()})

pd.set_option('display.width', 200)
print(len(ix), 'cells compared')
print(comp.round(3))

### Resolution test result

A 10 m version of the Sentinel features exceeded Earth Engine's memory limits even on a small test square, so features are built at **250 m** and averaged to the 2 km grid. To check this shortcut, the 250 m results were compared with a 100 m version on a 0.3-degree test square near Kano (289 grid cells):

- The means of all 20 features agree closely (e.g. September NDVI 0.511 at both resolutions).
- Correlations are 0.995-0.999 for 18 of 20 features. The lowest are `vhvv_wet` (0.989) and `vv_wet_std` (0.977), and the latter is sensitive to resolution, so treat it as a coarse indicator.
- This shows agreement with the 100 m version, not with a true 10 m result.

In [ ]:
# @title Cell 6: Full Export
# ---------- Full Sentinel export: submits tasks in manageable batches ----------
PREFIX       = 's4'
SCALE_M      = 250
BLOCK        = 0.5                 # degrees per task
LIMIT_TO     = None         # first run: Gambia only. Set to None for all countries
START        = True               # False = only print the plan; True = submit tasks
MAX_PENDING  = 900                 # most tasks waiting or running at the same time
RETRY_FAILED = False

ORDER = ['Gambia', 'Senegal', 'Mauritania', 'Mali', 'Burkina Faso',
         'Niger', 'Nigeria', 'Chad', 'Cameroon']

def state_of(t):
    s = getattr(t, 'state', None)
    return s if s else t.status()['state']

# 1. Group ALL grid cells into blocks; a block exports every cell inside it
g = grid[['grid_id', 'lon', 'lat', 'adm0']].copy()
g['ix'] = np.floor(g.lon / BLOCK).astype(int)
g['iy'] = np.floor(g.lat / BLOCK).astype(int)
g['rank'] = g.adm0.map({c: i for i, c in enumerate(ORDER)})

blocks = []
for (ix, iy), d in g.groupby(['ix', 'iy']):
    if LIMIT_TO and not d.adm0.isin(LIMIT_TO).any():
        continue
    blocks.append((int(d['rank'].min()), int(ix), int(iy), d))
blocks.sort(key=lambda b: (b[0], b[1], b[2]))
n_cells_scope = sum(len(b[3]) for b in blocks)
print(f'{len(blocks)} blocks, {n_cells_scope} grid cells -> {len(blocks) * 3} tasks in scope')

# 2. What has already been submitted?
rank_state = {'COMPLETED': 3, 'RUNNING': 2, 'READY': 2}
existing = {}
for t in ee.batch.Task.list():
    name = t.config.get('description', '')
    if name.startswith(PREFIX + '_'):
        s = state_of(t)
        if name not in existing or rank_state.get(s, 0) > rank_state.get(existing[name], 0):
            existing[name] = s
pending = sum(1 for s in existing.values() if s in ('READY', 'RUNNING'))
room = max(MAX_PENDING - pending, 0)

groups = {'ndvi': s2_ndvi, 'oth': s2_other, 's1': s1_feats}
todo, n_failed = [], 0
for _, ix, iy, d in blocks:
    for tag in groups:
        name = f'{PREFIX}_{tag}_x{ix + 200}_y{iy + 200}'
        st = existing.get(name)
        if st in ('COMPLETED', 'READY', 'RUNNING'):
            continue
        if st in ('FAILED', 'CANCELLED') and not RETRY_FAILED:
            n_failed += 1
            continue
        todo.append((name, tag, ix, iy, d))
print(f'{len(todo)} tasks still to submit, {pending} pending now, room for {room}, '
      f'{n_failed} failed/cancelled (not resubmitted)')

# 3. Submit
if START:
    submitted = 0
    for name, tag, ix, iy, d in todo[:room]:
        x0, y0 = ix * BLOCK, iy * BLOCK
        region = ee.Geometry.Rectangle([x0, y0, x0 + BLOCK, y0 + BLOCK])
        img = groups[tag](region, SCALE_M)
        ee.batch.Export.table.toDrive(
            collection=img.sampleRegions(collection=to_fc(d), properties=['grid_id'],
                                         scale=GRID_SCALE, projection=proj, tileScale=8),
            description=name, folder=OUT_FOLDER, fileFormat='CSV').start()
        submitted += 1
    print('Submitted', submitted, 'tasks.')
    if len(todo) > submitted:
        print('More remain: re-run this cell once tasks have finished to submit the next batch.')
    else:
        print('Everything in scope has now been submitted.')
else:
    print('START is False, so nothing was submitted.')

### How the full export runs (and how to re-run it)

The export cell submits one Earth Engine task per 0.5° block and feature group (NDVI, other Sentinel-2, Sentinel-1). In this project that is **919 blocks covering all 301,142 grid cells, or 2,757 tasks**. To keep the queue manageable, the cell never has more than `MAX_PENDING` (900) tasks waiting or running at once, so the full export is submitted in several batches.

**Settings in the cell**

| Setting | Meaning |
|---|---|
| `LIMIT_TO` | `['Gambia']` exports only blocks touching that country (used first, to measure compute cost on a small area). `None` covers every country. |
| `START` | `False` only prints the plan. `True` submits tasks. |
| `MAX_PENDING` | Maximum number of tasks queued or running at the same time. |
| `RETRY_FAILED` | `False` (default): failed or cancelled tasks are not resubmitted automatically. |

**To run it**

1. Run the cell with `START = False` and read the plan (blocks, grid cells, tasks).
2. Set `START = True` and run it. It submits up to `MAX_PENDING` tasks and reports how many remain.
3. Check progress with the monitor cell below. Earth Engine processes the queue on its own servers, so tasks keep running even if Colab disconnects.
4. When the queue has mostly drained, **run the export cell again**. It skips every task that is already completed, queued or running, tops the queue back up, and reports how many remain. Repeat until it prints "Everything in scope has now been submitted."
5. Once all tasks show `COMPLETED`, run the merge cell to build `sentinel.parquet`.

**Good to know**

- Re-running is safe: tasks are recognised by name, so nothing is submitted twice.
- Submitting new batches needs a live Colab session. After a disconnect, re-run the setup and builder cells first (they define `grid`, `to_fc`, `proj` and the feature builders), then the export cell.
- Failed tasks are listed by the monitor cell with their error message. Find the cause before setting `RETRY_FAILED = True`, so quota isn't spent on repeat failures.
- Earth Engine's compute quota resets monthly. Check usage in Cloud Monitoring (Metrics Explorer) before and after large batches.

In [ ]:
# @title Cell 7: merge into Sentinel.parquet
import glob, os
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
OUT_FOLDER = 'crop_project_sentinel'

def read_group(tag):
    files = glob.glob(f'{BASE}/{OUT_FOLDER}/s4_{tag}_*.csv')
    parts = [pd.read_csv(f) for f in files if os.path.getsize(f) > 0]
    d = (pd.concat(parts, ignore_index=True)
         .drop(columns=['system:index', '.geo'], errors='ignore')
         .drop_duplicates('grid_id'))
    print(f'{tag}: {len(files)} files, {len(d)} rows')
    return d.set_index('grid_id')

sen = pd.concat([read_group(t) for t in ['ndvi', 'oth', 's1']], axis=1, join='outer')
sen = sen.replace(-9999, np.nan)

# Derived vegetation-season features (computed here, not on Earth Engine)
nd = [f'ndvi_m{m:02d}' for m in range(1, 13)]
sen['ndvi_max'] = sen[nd].max(axis=1)
sen['ndvi_min'] = sen[nd].min(axis=1)
sen['ndvi_amp'] = sen['ndvi_max'] - sen['ndvi_min']
peak = sen[nd].idxmax(axis=1, skipna=True)
sen['ndvi_peak_month'] = pd.to_numeric(peak.astype(str).str[-2:], errors='coerce')
sen['ndvi_wet_mean'] = sen[['ndvi_m07', 'ndvi_m08', 'ndvi_m09', 'ndvi_m10']].mean(axis=1)
sen['ndvi_dry_mean'] = sen[['ndvi_m12', 'ndvi_m01', 'ndvi_m02', 'ndvi_m03']].mean(axis=1)

sen = sen.astype('float32').reset_index()

# Checks
missing = grid[~grid.grid_id.isin(sen.grid_id)]
print(f'\n{len(missing)} of {len(grid)} grid cells have no row yet')
feat = [c for c in sen.columns if c != 'grid_id']
print('Highest NaN shares (%):')
print((sen[feat].isna().mean() * 100).round(1).sort_values(ascending=False).head(8))

chk = grid[['grid_id', 'adm0']].merge(sen, on='grid_id')
print('\nMedian by country:')
print(chk.groupby('adm0')[['ndvi_max', 'ndvi_amp', 'vv_wet', 'si_dry']].median().round(2))

sen.to_parquet(f'{BASE}/crop_project/sentinel.parquet', index=False)
print('\nSaved', sen.shape)

MODIS NDVI trend:Captures change over time: whether an area has been greening or browning over 24 years.

In [ ]:
# @title Cell 8: builder and test
MODIS_YEARS = (2001, 2024)

def modis_features(region):
    col = (ee.ImageCollection('MODIS/061/MOD13Q1').filterBounds(region)
           .filter(ee.Filter.calendarRange(7, 10, 'month')))
    P = col.first().select('NDVI').projection()

    def yearly(y):
        y = ee.Number(y)
        im = (col.filter(ee.Filter.calendarRange(y, y, 'year'))
              .map(lambda i: i.select('NDVI').multiply(0.0001)
                              .updateMask(i.select('SummaryQA').lte(1)))   # good or marginal pixels
              .mean().rename('ndvi'))
        return im.addBands(ee.Image.constant(y).float().rename('year')).set('year', y)

    yc = ee.ImageCollection(ee.List.sequence(MODIS_YEARS[0], MODIS_YEARS[1]).map(yearly))
    mean = yc.select('ndvi').mean().rename('modis_gs_ndvi_mean')
    cv = (yc.select('ndvi').reduce(ee.Reducer.stdDev()).divide(mean)
          .rename('modis_gs_ndvi_cv'))
    trend = (yc.select(['year', 'ndvi']).reduce(ee.Reducer.linearFit())
             .select('scale').rename('modis_gs_ndvi_trend'))
    early = yc.filter(ee.Filter.lte('year', MODIS_YEARS[0] + 9)).select('ndvi').mean()
    late = yc.filter(ee.Filter.gte('year', MODIS_YEARS[1] - 9)).select('ndvi').mean()
    change = late.subtract(early).rename('modis_gs_ndvi_change')

    img = ee.Image.cat([mean, cv, trend, change]).float().setDefaultProjection(P)
    return (img.reduceResolution(ee.Reducer.mean(), maxPixels=1024)
               .reproject(proj).unmask(-9999))

# Test on 5 points near Kano
test = (grid[(grid.lon >= 8) & (grid.lon < 9) & (grid.lat >= 12) & (grid.lat < 13)]
        .sample(5, random_state=3))
reg = ee.Geometry.Rectangle([8, 12, 9, 13])
res = modis_features(reg).sampleRegions(collection=to_fc(test), properties=['grid_id'],
                                         scale=GRID_SCALE, projection=proj, tileScale=4).getInfo()
print(pd.DataFrame([f['properties'] for f in res['features']]).set_index('grid_id').round(4).T)

In [ ]:
# @title Cell 9: Cell B: export
BLOCK = 2
g = grid[['grid_id', 'lon', 'lat']].copy()
g['ix'] = np.floor(g.lon / BLOCK).astype(int)
g['iy'] = np.floor(g.lat / BLOCK).astype(int)

existing = {t.config.get('description') for t in ee.batch.Task.list()}
blocks = list(g.groupby(['ix', 'iy']))
print(len(blocks), 'blocks')

n = 0
for (ix, iy), d in blocks:
    ix, iy = int(ix), int(iy)                      # plain Python ints
    name = f's4m_x{ix + 200}_y{iy + 200}'
    if name in existing:
        continue
    x0, y0 = float(ix * BLOCK), float(iy * BLOCK)
    reg = ee.Geometry.Rectangle([x0, y0, x0 + BLOCK, y0 + BLOCK])
    ee.batch.Export.table.toDrive(
        collection=modis_features(reg).sampleRegions(collection=to_fc(d), properties=['grid_id'],
                                                     scale=GRID_SCALE, projection=proj, tileScale=4),
        description=name, folder=OUT_FOLDER, fileFormat='CSV').start()
    n += 1
print('Submitted', n, 'tasks')

In [ ]:
# @title Cell 10: merge into sentinel.parquet
files = glob.glob(f'{BASE}/{OUT_FOLDER}/s4m_*.csv')
mod = (pd.concat([pd.read_csv(f) for f in files if os.path.getsize(f) > 0], ignore_index=True)
       .drop(columns=['system:index', '.geo'], errors='ignore')
       .drop_duplicates('grid_id').replace(-9999, np.nan))
mod[[c for c in mod.columns if c != 'grid_id']] = mod[[c for c in mod.columns if c != 'grid_id']].astype('float32')
print(len(files), 'files ->', mod.shape)

sen = pd.read_parquet(f'{BASE}/crop_project/sentinel.parquet')
sen = sen.drop(columns=[c for c in sen.columns if c.startswith('modis_')], errors='ignore')
sen = sen.merge(mod, on='grid_id', how='left')
print('Cells without MODIS values:', sen['modis_gs_ndvi_mean'].isna().sum())
print(sen.filter(like='modis_').describe().T[['min', '50%', 'max']].round(4))
sen.to_parquet(f'{BASE}/crop_project/sentinel.parquet', index=False)
print('Saved', sen.shape)

In [ ]:
# @title Clean MODIS NDVI variability (CV). Sets modis_gs_ndvi_cv to NaN where mean NDVI < 0.1 or CV is outside 0-1 (~0.4% of cells),then re-saves sentinel.parquet. Safe to re-run.
sen = pd.read_parquet(f'{BASE}/crop_project/sentinel.parquet')
m, cv = sen['modis_gs_ndvi_mean'], sen['modis_gs_ndvi_cv']

low_mean = m < 0.1
odd_cv = ((cv < 0) | (cv > 1)) & ~low_mean
bad = low_mean | odd_cv
print('Cells with mean NDVI < 0.1:', int(low_mean.sum()))
print('Other cells with CV outside 0-1:', int(odd_cv.sum()))

where = grid[['grid_id', 'adm0']].merge(sen.loc[bad, ['grid_id']], on='grid_id')
print('\nAffected cells by country:'); print(where.adm0.value_counts())

sen.loc[bad, 'modis_gs_ndvi_cv'] = np.nan
cv2 = sen['modis_gs_ndvi_cv']
print(f'\nCV set to NaN for {int(bad.sum())} cells ({bad.mean() * 100:.2f}%)')
print('CV range after the fix:', round(float(cv2.min()), 3), 'to', round(float(cv2.max()), 3))

sen.to_parquet(f'{BASE}/crop_project/sentinel.parquet', index=False)
print('Saved', sen.shape)

In [ ]:
# @title Optional check: The five feature files should all contain exactly the same 301,142 grid_id values.
import os
files = ['grid', 'climate', 'soil_terrain', 'salinity', 'sentinel']
ref = set(grid.grid_id)
for name in files:
    path = f'{BASE}/crop_project/{name}.parquet'
    if not os.path.exists(path):
        print(f'{name}: FILE NOT FOUND'); continue
    d = pd.read_parquet(path)
    print(f'{name:13} rows={len(d):>7}  unique ids={d.grid_id.nunique():>7}  '
          f'same ids as grid={set(d.grid_id) == ref}  '
          f'columns={d.shape[1] - 1:>3}  max NaN share={d.drop(columns="grid_id").isna().mean().max() * 100:.1f}%')